# AutoGluon baselines: No Preprocessing and Default preprocessing

Both settings use the canonical 30-dataset suite and the repository's
shared `split_train_val_test` function. AutoGluon trains on outer train,
uses outer validation for selection, and scores untouched outer test once.

AutoGluon is limited to 300 seconds per dataset/setting in both modes;
smoke mode only reduces the number of datasets to one per shard.


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/MothMalone/SolutionRecommendation.git"
REPO_BRANCH = "feature/acorec-autodp-space"
REPO_DIR = Path("/kaggle/working/SolutionRecommendation")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements-kaggle.txt")], check=True)
os.chdir(REPO_DIR)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


In [ ]:
import gc, json, os, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(REPO_DIR / "src"))
from automl_aco.eval_ids import EVAL_DATASETS

RUN_MODE = "smoke"       # change to final after smoke succeeds
NUM_DATASET_SHARDS = 5
DATASET_SHARD_INDEX = 0
SPLIT_SEED = 42
TRAIN_SEED = 1
MAX_SAMPLES = 100_000
AG_TIME_LIMIT = 300
AG_PRESETS = "best_quality"
DATASETS = [{"dataset_id": int(value), "name": name} for name, value in EVAL_DATASETS.items()]
if RUN_MODE not in {"smoke", "final"} or not 0 <= DATASET_SHARD_INDEX < NUM_DATASET_SHARDS:
    raise ValueError("Invalid RUN_MODE or shard index")
positions = np.array_split(np.arange(len(DATASETS)), NUM_DATASET_SHARDS)
RUN_DATASETS = [DATASETS[int(i)] for i in positions[DATASET_SHARD_INDEX]]
if RUN_MODE == "smoke":
    RUN_DATASETS = RUN_DATASETS[:1]
BASE_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR = BASE_DIR / "autogluon_baselines"
CACHE_DIR = BASE_DIR / "openml_datagit_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True); CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Shard {DATASET_SHARD_INDEX}/{NUM_DATASET_SHARDS - 1}; limit={AG_TIME_LIMIT}s; ids={[x['dataset_id'] for x in RUN_DATASETS]}")


In [ ]:
evaluator_path = REPO_DIR / "scripts" / "autogluon_evaluator.py"
if not evaluator_path.exists():
    raise RuntimeError("Remote checkout is stale: scripts/autogluon_evaluator.py is missing. Push the AutoGluon scripts to feature/acorec-autodp-space, then restart this Kaggle session.")
evaluator_source = evaluator_path.read_text(encoding="utf-8")
if "def evaluate_autogluon_split" not in evaluator_source or "IdentityFeatureGenerator" not in evaluator_source:
    raise RuntimeError("Stale AutoGluon evaluator; restart the Kaggle session and rerun setup")
import autogluon.tabular
print("AutoGluon loaded")


In [ ]:
RESULT_PATH = OUTPUT_DIR / f"autogluon_baselines_shard_{DATASET_SHARD_INDEX:02d}_of_{NUM_DATASET_SHARDS:02d}.csv"
RESULT_DIR = OUTPUT_DIR / "per_dataset"; RESULT_DIR.mkdir(parents=True, exist_ok=True)
SETTINGS = [("no_preprocessing", "identity"), ("default_preprocessing", "default")]
rows = pd.read_csv(RESULT_PATH).to_dict("records") if RESULT_PATH.exists() else []
def upsert(row):
    key = (str(row.get("dataset_id")), str(row.get("setting")))
    rows[:] = [old for old in rows if (str(old.get("dataset_id")), str(old.get("setting"))) != key]
    rows.append(row); pd.DataFrame(rows).to_csv(RESULT_PATH, index=False)
for position, spec in enumerate(RUN_DATASETS, start=1):
    for setting, feature_generator in SETTINGS:
        if any(str(x.get("dataset_id")) == str(spec["dataset_id"]) and x.get("setting") == setting and x.get("status") == "ok" for x in rows):
            print("SKIP successful:", spec["name"], setting); continue
        output_json = RESULT_DIR / f"{int(spec['dataset_id'])}_{setting}.json"
        started_at = datetime.now(timezone.utc).isoformat(); started = time.perf_counter()
        command = [sys.executable, str(REPO_DIR / "scripts/evaluate_autogluon_baseline.py"), "--dataset-id", str(spec["dataset_id"]), "--dataset-name", spec["name"], "--data-dir", str(CACHE_DIR), "--output-json", str(output_json), "--setting", setting, "--feature-generator", feature_generator, "--split-seed", str(SPLIT_SEED), "--time-limit", str(AG_TIME_LIMIT), "--presets", AG_PRESETS, "--max-samples", str(MAX_SAMPLES)]
        print(f"[{position}/{len(RUN_DATASETS)}] {spec['name']} / {setting}")
        process = subprocess.run(command, cwd=REPO_DIR, check=False)
        row = json.loads(output_json.read_text(encoding="utf-8")) if output_json.exists() else {"status": "failed", "error": f"exit code {process.returncode}"}
        row.update({"dataset_id": int(spec["dataset_id"]), "dataset": spec["name"], "setting": setting, "feature_generator": feature_generator, "started_at_utc": started_at, "finished_at_utc": datetime.now(timezone.utc).isoformat(), "notebook_wall_clock_seconds": time.perf_counter() - started})
        upsert(row); gc.collect()
display(pd.DataFrame(rows).sort_values(["dataset_id", "setting"]))
print("Saved:", RESULT_PATH)


`fit_seconds`, `prediction_seconds`, `total_seconds`, and
`notebook_wall_clock_seconds` are stored for every dataset/setting.
The final test is never supplied during fitting or model selection.
